# **04_GOLD — Modelado Analítico (Capa Gold)**

## Descripción
Esta notebook implementa la capa Gold de la arquitectura Medallón.
A partir de las tablas limpias y estandarizadas en Silver, se cargan, 
tipan y persisten las dimensiones y la tabla de hechos en formato Delta, 
dejando el modelo estrella completo y disponible para consumo analítico.

## Criterios de la Capa Gold
| Criterio | Descripción |
|---|---|
| Tipos de dato correctos | Todas las columnas casteadas explícitamente |
| Modelo estrella completo | 3 dimensiones + 1 tabla de hechos relacionadas |
| Formato Delta | Persistencia optimizada para consultas analíticas |
| Idempotencia | Modo `overwrite` permite re-ejecución sin duplicados |

## Tablas Generadas
| Tabla | Tipo | Descripción |
|---|---|---|
| `dim_categoria` | Dimensión | Categorías únicas de producto |
| `dim_fecha` | Dimensión | Calendario con atributos temporales |
| `dim_genero` | Dimensión | Géneros normalizados de clientes |
| `hechos_ventas` | Hechos | Transacciones con métricas y claves foráneas |

## Flujo de la Notebook
1. Carga y tipado de `dim_categoria` desde Silver
2. Carga y tipado de `dim_fecha` desde Silver
3. Carga, normalización y tipado de `dim_genero` desde Silver
4. Carga y tipado de `hechos_ventas` desde Silver
5. Persistencia de todas las tablas en Gold como Delta


## **Verificación Final — hechos_ventas en Gold**
Carga y visualización de la tabla de hechos directamente desde Gold 
para confirmar que la persistencia fue exitosa y los datos están 
disponibles para consumo analítico.

In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

hechos_ventas = spark.read.table("silver.dbo.hechos_ventas")
display(hechos_ventas)

StatementMeta(, 49a3e17f-06a2-4127-8a04-231344468664, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bbad1dc2-bde2-4682-88b3-072f602fd366)

## **Verificación Final — dim_categoria en Gold**
Carga y visualización de la dimensión directamente desde Gold 
para confirmar que la persistencia fue exitosa y los datos están 
disponibles para consumo analítico.

In [2]:
dim_categoria = spark.read.table("silver.dbo.dim_categoria")
display(dim_categoria)

StatementMeta(, 49a3e17f-06a2-4127-8a04-231344468664, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1ad104dc-092a-4f74-bfec-b1ddc9ae54f7)

## **Verificación Final — dim_fecha en Gold**
Carga y visualización de la dimensión calendario directamente desde Gold 
para confirmar que la persistencia fue exitosa y los datos están 
disponibles para consumo analítico.

In [3]:
dim_fecha= spark.read.table("silver.dbo.dim_fecha")
display(dim_fecha)

StatementMeta(, 49a3e17f-06a2-4127-8a04-231344468664, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 395d8eb9-73c4-4882-b71d-7d57db442586)

## **Verificación Final — dim_genero en Gold**
Carga y visualización de la dimensión de géneros directamente desde Gold 
para confirmar que la persistencia fue exitosa y los datos están 
disponibles para consumo analítico.

In [4]:
dim_genero = spark.read.table("silver.dbo.dim_genero")
display(dim_genero)

StatementMeta(, 49a3e17f-06a2-4127-8a04-231344468664, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5aaffdf4-5528-4678-afb6-94583a6c3138)

## **Consulta 1 — Ranking de Categorías por Ingresos**
Identifica qué categoría de producto genera más ingresos para RetailNova,
permitiendo enfocar estrategias comerciales en los segmentos más rentables.

**Técnica:** `JOIN` con `dim_categoria` + `GROUP BY` + `RANK()` descendente  
**Pregunta de negocio:** ¿Cuáles son las categorías con mayor venta?

In [5]:
from pyspark.sql.functions import sum, rank
from pyspark.sql.window import Window
from pyspark.sql.functions import col

resultado1 = hechos_ventas \
    .join(dim_categoria, "id_categoria") \
    .groupBy("product_category") \
    .agg(sum("total_amount").alias("total_ingresos")) \
    .withColumn("ranking", rank().over(Window.orderBy(col("total_ingresos").desc())))

display(resultado1)

StatementMeta(, 49a3e17f-06a2-4127-8a04-231344468664, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 496041b0-4b88-4d33-9bb5-87fb4cd98b9f)

## **Consulta 2 — Clientes Frecuentes vs Clientes Únicos**
Clasifica a los clientes según su frecuencia de compra para identificar 
el nivel de fidelización y orientar estrategias de retención.

**Técnica:** `GROUP BY` + `COUNT(transaction_id)` + `CASE WHEN` para clasificación  
**Pregunta de negocio:** ¿Qué clientes generan más ingresos y con qué frecuencia compran?

In [6]:
from pyspark.sql.functions import count, when

resultado2 = hechos_ventas \
    .groupBy("customer_id") \
    .agg(count("transaction_id").alias("total_compras")) \
    .withColumn("tipo_cliente", when(col("total_compras") > 1, "Frecuente").otherwise("Unico")) \
    .orderBy(col("total_compras").desc())

display(resultado2)

StatementMeta(, 49a3e17f-06a2-4127-8a04-231344468664, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fc418e6d-2b67-4155-9d88-b343c32a7ef6)

## **Consulta 3 — Evolución de Ventas por Mes**
Analiza la tendencia mensual de ventas para detectar estacionalidad,
picos de demanda y períodos de baja actividad comercial.

**Técnica:** `JOIN` con `dim_fecha` + `GROUP BY anio, mes` para serie temporal  
**Pregunta de negocio:** ¿Cómo evolucionan las ventas en el tiempo?

In [7]:
resultado3 = hechos_ventas \
    .join(dim_fecha, "id_fecha") \
    .groupBy("anio", "mes") \
    .agg(sum("total_amount").alias("total_ventas")) \
    .orderBy("anio", "mes")

display(resultado3)

StatementMeta(, 49a3e17f-06a2-4127-8a04-231344468664, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 38a18c4c-603c-45ff-8309-fd4f5bf82fe4)

## **Consulta 4 — Ticket Promedio por Género**
Compara el valor promedio de compra entre géneros para identificar 
diferencias de comportamiento y orientar estrategias de pricing 
y marketing diferenciado.

**Técnica:** `JOIN` con `dim_genero` + `GROUP BY nombre_genero` + `AVG(total_amount)`  
**Pregunta de negocio:** ¿Cómo se comportan los géneros en términos de gasto promedio?

In [8]:
from pyspark.sql.functions import avg, round

resultado4 = hechos_ventas \
    .join(dim_genero, "id_genero") \
    .groupBy("nombre_genero") \
    .agg(round(avg("total_amount"), 2).alias("ticket_promedio"))

display(resultado4)

StatementMeta(, 49a3e17f-06a2-4127-8a04-231344468664, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5bf64129-6433-4c80-bba9-c9073ed2b280)

## **Consulta 5 — Día del Mes con Más Ventas**
Identifica qué días del mes concentran mayor actividad comercial 
para planificar promociones, refuerzos de inventario y campañas 
en los momentos de mayor demanda.

**Técnica:** `JOIN` con `dim_fecha` + `GROUP BY dia` + `SUM(total_amount)` ordenado descendente  
**Pregunta de negocio:** ¿Qué patrones de compra existen dentro del mes?

In [9]:
resultado5 = hechos_ventas \
    .join(dim_fecha, "id_fecha") \
    .groupBy("dia") \
    .agg(
        sum("total_amount").alias("total_ventas"),
        count("transaction_id").alias("total_transacciones")
    ) \
    .orderBy(col("total_ventas").desc())

display(resultado5)

StatementMeta(, 49a3e17f-06a2-4127-8a04-231344468664, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 73b3b4a1-3a35-4217-9716-d728eee355d3)